# 🔬 MSU AI Club — Workshop 04
## Bayesian Logistic Regression & Stochastic Search Variable Selection (SSVS)
### Case Study: Modeling Odds of Malignancy in Breast Cancer Biopsies (Attendee Handout)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue.svg)](https://www.python.org/)
[![Status: Attendee Handout](https://img.shields.io/badge/Status-Attendee%20Handout-green.svg)](https://github.com)

---

### 📌 Workshop Overview & Learning Objectives
Welcome to the interactive coding laboratory for **Workshop 04: Bayesian Statistical Methods & Machine Learning**!

In high-stakes domains like clinical oncology, standard machine learning classifiers that only produce hard 0/1 predictions or uncalibrated confidence scores are insufficient. Medical professionals require:
1. **Calibrated Posterior Uncertainty**: Knowing when the model is confident vs. when a biopsy is borderline.
2. **Clinical Odds Ratios**: Interpretable biometric multipliers (e.g., "each 1-SD increase in worst nuclear radius multiplies odds of malignancy by $12\times$").
3. **Sparse Biometric Feature Selection**: Discovering which cell nuclei characteristics are truly predictive vs. redundant noise.

### 🎯 Your Mission Today:
Work through this interactive notebook with your pair-programming partner to complete the **7 `# TODO:` challenge slots**:
* **TODO 1**: Binary Target Encoding (`M` $\to 1$, `B` $\to 0$)
* **TODO 2**: Numerically Stable Logistic Sigmoid Function
* **TODO 3**: Bernoulli Log-Likelihood & Log-Posterior Formulation
* **TODO 4**: Metropolis-Hastings Acceptance Condition & Decision Rule
* **TODO 5**: Pilot Chain Proposal Tuning (The Goldilocks Principle)
* **TODO 6**: Posterior Inclusion Probabilities (PIP) via Stochastic Search
* **TODO 7**: Posterior Odds Ratios & Clinical Oncology Reflection

---
**Interactive Student Copy**

---
## 1. Clinical Context: The Wisconsin Diagnostic Breast Cancer (WDBC) Dataset

We analyze the **Wisconsin Diagnostic Breast Cancer (WDBC)** dataset (*Wolberg et al., 1993*, University of Wisconsin / UCI Machine Learning Repository). 

The dataset contains quantitative morphological measurements computed from digitized images of **Fine Needle Aspirate (FNA)** biopsies across **569 patients**:
* **Benign ($B$)**: 357 patients (62.7%)
* **Malignant ($M$)**: 212 patients (37.3%)

### The 30 Nuclear Characteristics
For each cell nucleus, 10 fundamental geometric traits were measured:
1. **Radius**: Distance from center to perimeter points
2. **Texture**: Standard deviation of gray-scale pixel intensities
3. **Perimeter**: Outer boundary length of the nucleus
4. **Area**: Nuclear cross-sectional surface area
5. **Smoothness**: Local radius length variation
6. **Compactness**: Computed as $\frac{\text{perimeter}^2}{\text{area}} - 1.0$
7. **Concavity**: Severity of concave indentations along the contour
8. **Concave Points**: Number of concave indentations
9. **Symmetry**: Nuclear symmetry
10. **Fractal Dimension**: "Coastline approximation" error $- 1.0$

For each patient, these 10 traits are reported across 3 statistical aggregations, yielding **30 total features**:
* **Suffix `1`**: Mean value across all observed nuclei (e.g. `radius1`)
* **Suffix `2`**: Standard error across observed nuclei (e.g. `radius2`)
* **Suffix `3`**: "Worst" (mean of the three largest values) (e.g. `radius3`)

In [ ]:
# Core scientific computing and visualization
import os
import io
import urllib.request
import numpy as np
import scipy.stats as stats
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.figsize": (9, 5)})
np.random.seed(42)

print("Environment successfully initialized!")

---
## 2. Data Ingestion & Design Matrix Construction

Let's load the dataset from the local `data/wdbc.data` path, or fall back to UCI download if running in Colab.

In [ ]:
def load_wdbc_data():
    local_path = "data/wdbc.data"
    base_metrics = ['radius', 'texture', 'perimeter', 'area', 'smoothness', 
                    'compactness', 'concavity', 'concave_points', 'symmetry', 'fractal_dimension']
    col_names = ['id', 'diagnosis'] + [f"{m}1" for m in base_metrics] + [f"{m}2" for m in base_metrics] + [f"{m}3" for m in base_metrics]
    
    if os.path.exists(local_path):
        print(f"Loading local dataset from: {local_path}")
        df = pd.read_csv(local_path, header=None, names=col_names)
    else:
        print("Local file not found. Downloading from UCI Machine Learning Repository...")
        uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
        req = urllib.request.Request(uci_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp:
            df = pd.read_csv(io.BytesIO(resp.read()), header=None, names=col_names)
        print("Download complete!")
    return df

wdbc_df = load_wdbc_data()
print(f"Loaded {wdbc_df.shape[0]} patients and {wdbc_df.shape[1]} columns.")
wdbc_df[['id', 'diagnosis', 'radius1', 'texture1', 'concave_points1', 'radius3']].head()

### ✏️ Exercise 1: Binary Target Encoding
Our clinical response variable is `diagnosis`, containing strings `'M'` (Malignant) and `'B'` (Benign). 

For binary logistic regression, we model the probability of malignancy:
$$y_i = \begin{cases} 1 & \text{if biopsy is Malignant } ('M') \\ 0 & \text{if biopsy is Benign } ('B') \end{cases}$$

#### 💡 Your Task:
Fill in the code below to convert the `'diagnosis'` column into a 1D NumPy array `y` of integers ($0$ and $1$).

In [ ]:
# ==============================================================================
# TODO 1: Binary Target Encoding
# Convert wdbc_df['diagnosis'] to a 1D NumPy array of ints: 1 for 'M', 0 for 'B'
# ==============================================================================
y = None  # <-- YOUR CODE HERE

# Sanity checks
assert len(y) == 569, "Length must be 569 patients"
assert set(np.unique(y)) == {0, 1}, "Values must be exactly 0 and 1"
assert np.sum(y) == 212, "There must be exactly 212 malignant cases"
print(f"Successfully encoded target: {np.sum(y == 0)} Benign (0), {np.sum(y == 1)} Malignant (1)")

In [ ]:
# Extract the 30 continuous feature columns (discard id and diagnosis)
feature_cols = [c for c in wdbc_df.columns if c not in ['id', 'diagnosis']]
X_raw = wdbc_df[feature_cols].values
n, p_feat = X_raw.shape

# Standardize predictors to zero mean and unit variance: (X - mean) / std
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0, ddof=1)
X_scaled = (X_raw - X_mean) / X_std

# Bind column of ones for the intercept term b0
X = np.column_stack([np.ones(n), X_scaled])
param_names = ['(Intercept)'] + feature_cols
p = len(param_names)

print(f"Design matrix X shape: {X.shape} (n={n} patients, p={p} parameters)")

---
## 3. Mathematical Foundations: Likelihood, Prior, and the Sigmoid Function

### The Logistic Model
For each patient $i = 1, \dots, n$, the probability of malignancy is modeled via the logistic sigmoid link:
$$\theta_i = P(y_i = 1 \mid \mathbf{x}_i, \boldsymbol{eta}) = \sigma(\mathbf{x}_i^T\boldsymbol{eta}) = \frac{1}{1 + e^{-\mathbf{x}_i^T\boldsymbol{eta}}} = \frac{e^{\mathbf{x}_i^T\boldsymbol{eta}}}{1 + e^{\mathbf{x}_i^T\boldsymbol{eta}}}$$

Equivalently, on the log-odds (logit) scale:
$$\text{logit}(\theta_i) = \log\left(\frac{\theta_i}{1 - \theta_i}\right) = \mathbf{x}_i^T\boldsymbol{eta} = \beta_0 + \sum_{j=1}^{30} \beta_j x_{ij}$$

---

### ✏️ Exercise 2: Numerically Stable Sigmoid Function
In floating-point arithmetic, calculating $e^{-z}$ for large negative numbers (e.g. $z = -1000$) can cause numerical overflow (`RuntimeWarning: overflow in exp`). 

To make our implementation robust, we clip the input $z$ to a safe range $[-500, 500]$ before computing the exponential:
$$\sigma(z) = \frac{1}{1 + \exp(-\text{clip}(z, -500, 500))}$$

#### 💡 Your Task:
Implement the `sigmoid` function below.

In [ ]:
# ==============================================================================
# TODO 2: Numerically Stable Sigmoid Function
# ==============================================================================
def sigmoid(z):
    """
    Computes the logistic sigmoid 1 / (1 + exp(-z)).
    Clip z to [-500.0, 500.0] using np.clip to prevent floating-point overflow.
    """
    # <-- YOUR CODE HERE
    pass

# Sanity checks
assert np.isclose(sigmoid(0.0), 0.5), "sigmoid(0) must equal 0.5"
assert np.isclose(sigmoid(1000.0), 1.0), "sigmoid(1000) must equal 1.0"
assert np.isclose(sigmoid(-1000.0), 0.0), "sigmoid(-1000) must equal 0.0"
print("Sigmoid function test passed successfully!")

### The Bernoulli Likelihood & Prior Distribution
Assuming observations are independent, the joint Bernoulli likelihood is:
$$p(\mathbf{y} \mid \mathbf{X}, \boldsymbol{eta}) = \prod_{i=1}^n \theta_i^{y_i} (1 - \theta_i)^{1 - y_i} = \prod_{i=1}^n \frac{e^{y_i \mathbf{x}_i^T\boldsymbol{eta}}}{1 + e^{\mathbf{x}_i^T\boldsymbol{eta}}}$$

Taking the natural logarithm converts products into sums:
$$\log p(\mathbf{y} \mid \mathbf{X}, \boldsymbol{eta}) = \sum_{i=1}^n \left[ y_i (\mathbf{x}_i^T\boldsymbol{eta}) - \log(1 + e^{\mathbf{x}_i^T\boldsymbol{eta}}) \right]$$

#### Weakly Informative Gaussian Prior
We adopt a diffuse, weakly informative Gaussian prior centered at zero:
$$\boldsymbol{eta} \sim \mathcal{N}(\mathbf{0}, \sigma_0^2 \mathbf{I}_p), \quad \text{with } \sigma_0^2 = 100$$
$$\log p(\boldsymbol{eta}) = -\frac{p}{2}\log(2\pi\sigma_0^2) - \frac{1}{2\sigma_0^2} \sum_{j=0}^{30} \beta_j^2$$

---

### ✏️ Exercise 3: Bernoulli Log-Likelihood & Log-Posterior
#### 💡 Your Task:
Complete the `calc_log_likelihood` function below. 
*Hint*: You can compute $\log(1 + e^z)$ with maximal numerical precision using NumPy's `np.logaddexp(0, z)`!

In [ ]:
# ==============================================================================
# TODO 3: Bernoulli Log-Likelihood
# ==============================================================================
sigma0_sq = 100.0  # Weakly informative prior variance

def calc_log_likelihood(beta, X_mat, y_vec):
    """
    Calculates sum_{i=1}^n [ y_i * (x_i^T beta) - log(1 + exp(x_i^T beta)) ].
    Use np.logaddexp(0.0, xb) for log(1 + exp(xb)).
    """
    xb = X_mat @ beta
    # <-- YOUR CODE HERE
    pass

def calc_log_prior(beta):
    """Calculates log prior density: -0.5 / sigma0^2 * sum(beta^2)."""
    return -0.5 / sigma0_sq * np.sum(beta**2)

def calc_log_posterior(beta, X_mat, y_vec):
    """Unnormalized log-posterior: log-likelihood + log-prior."""
    return calc_log_likelihood(beta, X_mat, y_vec) + calc_log_prior(beta)

# Sanity check at beta = 0
beta_zero = np.zeros(p)
ll_zero = calc_log_likelihood(beta_zero, X, y)
expected_ll_zero = -len(y) * np.log(2.0)
assert np.isclose(ll_zero, expected_ll_zero), f"Expected {expected_ll_zero:.2f}, got {ll_zero:.2f}"
print(f"Log-likelihood at beta=0: {ll_zero:.2f} (Expected: {expected_ll_zero:.2f})")

---
## 4. The Metropolis-Hastings MCMC Algorithm

### Why MCMC? The Breakdown of Conjugacy
In Workshop 04's linear regression demo, Gaussian priors combined with a Gaussian likelihood to produce an exact conjugate Gaussian/Inverse-Gamma posterior. 

In logistic regression, the Bernoulli likelihood $p(\mathbf{y} \mid \boldsymbol{eta})$ is **non-conjugate** to the Gaussian prior $p(\boldsymbol{eta})$. The normalizing constant $\int p(\mathbf{y} \mid \boldsymbol{eta}) p(\boldsymbol{eta}) d\boldsymbol{eta}$ has no closed-form solution. We must use **Markov Chain Monte Carlo (MCMC)**!

### Symmetric Proposal Mechanism
At iteration $k$, we propose a candidate $\boldsymbol{eta}^*$ from a symmetric multivariate normal centered at the current state:
$$\boldsymbol{eta}^* \sim \mathcal{N}\left(\boldsymbol{eta}^{(k)}, \; \boldsymbol{\Sigma}_{\text{prop}}\right), \quad \text{where } \boldsymbol{\Sigma}_{\text{prop}} = \sigma_{\text{prop}}^2 (\mathbf{X}^T\mathbf{X})^{-1}$$

* **Why $(\mathbf{X}^T\mathbf{X})^{-1}$?** By shaping the proposal covariance to match the feature correlation structure, the algorithm makes coordinated moves along the curved ridges of the posterior!

### Acceptance Probability
Because the proposal is symmetric ($q(\boldsymbol{eta}^* \mid \boldsymbol{eta}) = q(\boldsymbol{eta} \mid \boldsymbol{eta}^*)$), the acceptance ratio on the log scale is simply:
$$\log r = \log p(\boldsymbol{eta}^* \mid \mathbf{y}) - \log p(\boldsymbol{eta}^{(k)} \mid \mathbf{y})$$
$$\alpha = \min(1, r) = \min(1, e^{\log r})$$

We accept $\boldsymbol{eta}^*$ if a random draw $u \sim \text{Uniform}(0, 1)$ satisfies $u \le \alpha$, or equivalently $\log u \le \log r$.

---

### ✏️ Exercise 4: Metropolis-Hastings Acceptance Step
#### 💡 Your Task:
Implement the acceptance condition in the code block below.

In [ ]:
# ==============================================================================
# TODO 4: Metropolis-Hastings Acceptance Rule
# ==============================================================================
def metropolis_step(beta_curr, log_post_curr, L_prop, X_mat, y_vec, rng):
    """
    Executes a single Metropolis-Hastings proposal and accept/reject decision.
    """
    p_dim = len(beta_curr)
    beta_prop = beta_curr + L_prop @ rng.standard_normal(p_dim)
    log_post_prop = calc_log_posterior(beta_prop, X_mat, y_vec)
    
    # <-- YOUR CODE HERE: Compute log_r and implement accept/reject rule
    # log_r = ...
    # if np.log(rng.uniform(0.0, 1.0)) <= log_r: ...
    pass

# Test step execution
inv_XtX = np.linalg.inv(X.T @ X)
L_test = np.linalg.cholesky(1.0 * inv_XtX)
b_next, lp_next, acc = metropolis_step(np.zeros(p), calc_log_posterior(np.zeros(p), X, y), L_test, X, y, np.random.default_rng(42))
print(f"Test step completed! Accepted: {acc}, Log-posterior: {lp_next:.2f}")

---
## 5. Pilot Chains & Proposal Tuning: The Goldilocks Principle

Before running 20,000 MCMC iterations, we must tune the proposal scale $\sigma_{\text{prop}}^2$. 

The MCMC sampler faces a fundamental trade-off:
* **Step size too small** ($\sigma_{\text{prop}}^2 \to 0$): The chain accepts almost every proposal ($>80\%$), but takes microscopic steps. It explores the parameter space at a snail's pace and suffers high autocorrelation.
* **Step size too large** ($\sigma_{\text{prop}}^2 \gg 10$): The chain proposes extreme parameter jumps that land in low-probability tails, resulting in near-total rejection ($<5\%$) and getting "stuck".
* **The Goldilocks Zone**: Seminal theoretical results by Gelman, Roberts, and Gilks (1996) prove that for high-dimensional multivariate Gaussian proposals ($p > 10$), the optimal acceptance rate is approximately **$23.4\% - 40\%$**!

Let's run pilot chains of 2,000 iterations across a grid of candidate variances:
$$\sigma_{\text{prop}}^2 \in \{0.5, \, 1.0, \, 2.0, \, 5.0, \, 10.0, \, 25.0, \, 50.0, \, 100.0\}$$

In [ ]:
# Run pilot chain grid search
candidate_sigmas = [0.5, 1.0, 2.0, 5.0, 10.0, 25.0, 50.0, 100.0]
pilot_results = []
inv_XtX = np.linalg.inv(X.T @ X)

for s2_val in candidate_sigmas:
    rng = np.random.default_rng(42)
    cov_cand = s2_val * inv_XtX
    # Symmetrize for Cholesky stability
    cov_cand = 0.5 * (cov_cand + cov_cand.T)
    L_cand = np.linalg.cholesky(cov_cand)
    
    b_state = np.zeros(p)
    lp_state = calc_log_posterior(b_state, X, y)
    n_acc = 0
    
    for _ in range(2000):
        b_state, lp_state, did_acc = metropolis_step(b_state, lp_state, L_cand, X, y, rng)
        if did_acc:
            n_acc += 1
            
    acc_rate = n_acc / 2000.0
    pilot_results.append({'sigma2_prop': s2_val, 'acceptance_rate': acc_rate})

pilot_df = pd.DataFrame(pilot_results)
print("=== PILOT CHAIN TUNING RESULTS (2,000 iterations each) ===")
pilot_df.round(3)

### ✏️ Exercise 5: Pilot Chain Analysis & Tuning Selection
#### 💡 Your Task:
Inspect the table above. Which candidate value of $\sigma_{\text{prop}}^2$ yields an acceptance rate closest to the clinical paper's target of $\approx 31.5\%$ (the center of the optimal 20–50% efficiency window)? 

Set `optimal_sigma2_prop` below.

In [ ]:
# ==============================================================================
# TODO 5: Tuning Selection
# Choose the value from candidate_sigmas whose acceptance rate is closest to 35%
# ==============================================================================
optimal_sigma2_prop = None  # <-- YOUR CODE HERE

print(f"Selected optimal proposal variance: {optimal_sigma2_prop}")
assert optimal_sigma2_prop == 5.0, "Optimal value should be 5.0" 

---
## 6. Stochastic Search Variable Selection (SSVS) across 30 Features

In clinical breast cytology, having 30 continuous features creates significant collinearity (e.g. `radius`, `perimeter`, and `area` all measure nuclear size). 

Including redundant features increases noise, risks overfitting, and prevents pathologists from identifying the primary biometric indicators.

### The Indicator Model Formulation
We introduce a latent binary vector $\mathbf{z} = (z_1, \dots, z_{30}) \in \{0, 1\}^{30}$:
* $z_j = 1$: Feature $j$ is active in the diagnostic model
* $z_j = 0$: Feature $j$ is excluded ($b_j$ is zeroed out)

The linear predictor becomes:
$$\text{logit}(\theta_i) = b_0 + \sum_{j=1}^{30} z_j b_j x_{ij}$$

The **effective coefficient vector** is $\boldsymbol{\beta}_{\text{eff}} = [b_0, \, z_1 b_1, \, z_2 b_2, \dots, z_{30} b_{30}]^T$.

### The Two-Step Metropolis-within-Gibbs Sampler
At each MCMC iteration $k$:
1. **Update Indicators $\mathbf{z}$ given current coefficients $\mathbf{b}$**:
   Propose $\mathbf{z}^*$ by flipping each indicator with probability $p_{\text{flip}} = 0.2$. Accept with log-likelihood ratio $\Delta_z = \log p(\mathbf{y} \mid \mathbf{X}, \mathbf{z}^*, \mathbf{b}) - \log p(\mathbf{y} \mid \mathbf{X}, \mathbf{z}^{(k)}, \mathbf{b})$.
2. **Update Coefficients $\mathbf{b}$ given updated indicators $\mathbf{z}^{(k+1)}$**:
   Propose $\mathbf{b}^* \sim \mathcal{N}\left(\mathbf{b}^{(k)}, \, 0.5(\mathbf{X}^T\mathbf{X})^{-1}\right)$. Accept with posterior ratio $\Delta_b$.

In [ ]:
def run_ssvs_sampler(X_mat, y_vec, n_iter=10000, burnin=2000, random_state=42):
    """Runs Stochastic Search Variable Selection for Bayesian Logistic Regression."""
    rng = np.random.default_rng(random_state)
    n_obs, n_params = X_mat.shape
    p_feat = n_params - 1
    
    # Pre-compute Cholesky factor for coefficient proposal
    cov_b = 0.5 * np.linalg.inv(X_mat.T @ X_mat)
    cov_b = 0.5 * (cov_b + cov_b.T)
    L_b = np.linalg.cholesky(cov_b)
    
    # State initialization
    b_curr = np.zeros(n_params)
    z_curr = np.ones(p_feat, dtype=int)
    
    beta_eff = np.concatenate([[b_curr[0]], z_curr * b_curr[1:]])
    ll_curr = calc_log_likelihood(beta_eff, X_mat, y_vec)
    
    # Storage for post-burnin draws
    post_samples = n_iter - burnin
    z_chain = np.zeros((post_samples, p_feat), dtype=int)
    b_chain = np.zeros((post_samples, n_params))
    beta_eff_chain = np.zeros((post_samples, n_params))
    
    for s in range(n_iter):
        # 1. Update indicator vector z
        flip_mask = rng.uniform(0.0, 1.0, p_feat) < 0.2
        z_prop = z_curr.copy()
        z_prop[flip_mask] = 1 - z_prop[flip_mask]
        
        beta_eff_prop = np.concatenate([[b_curr[0]], z_prop * b_curr[1:]])
        ll_prop = calc_log_likelihood(beta_eff_prop, X_mat, y_vec)
        
        if np.log(rng.uniform(0.0, 1.0)) <= (ll_prop - ll_curr):
            z_curr = z_prop
            ll_curr = ll_prop
            beta_eff = beta_eff_prop
            
        # 2. Update continuous coefficients b
        b_prop = b_curr + L_b @ rng.standard_normal(n_params)
        beta_eff_b_prop = np.concatenate([[b_prop[0]], z_curr * b_prop[1:]])
        ll_b_prop = calc_log_likelihood(beta_eff_b_prop, X_mat, y_vec)
        lp_b_prop = calc_log_prior(b_prop)
        lp_b_curr = calc_log_prior(b_curr)
        
        log_rb = (ll_b_prop + lp_b_prop) - (ll_curr + lp_b_curr)
        if np.log(rng.uniform(0.0, 1.0)) <= log_rb:
            b_curr = b_prop
            ll_curr = ll_b_prop
            beta_eff = beta_eff_b_prop
            
        # Store post-burnin draws
        if s >= burnin:
            idx = s - burnin
            z_chain[idx] = z_curr
            b_chain[idx] = b_curr
            beta_eff_chain[idx] = beta_eff
            
    return z_chain, b_chain, beta_eff_chain

print("Sampling 10,000 MCMC iterations with SSVS (burn-in: 2,000)...")
z_draws, b_draws, beta_eff_draws = run_ssvs_sampler(X, y, n_iter=10000, burnin=2000)
print("SSVS sampling completed successfully!")

### ✏️ Exercise 6: Posterior Inclusion Probabilities (PIP)
The **Posterior Inclusion Probability (PIP)** represents the marginal probability that feature $j$ belongs in the true cancer prediction model, averaging over all $2^{30} \approx 1.07 \text{ billion}$ candidate models:
$$\text{PIP}_j = P(z_j = 1 \mid \mathbf{y}, \mathbf{X}) \approx \frac{1}{S} \sum_{s=1}^S z_j^{(s)}$$

#### 💡 Your Task:
1. Compute the empirical mean of the indicator chain `z_draws` along axis 0.
2. Filter the features that exceed the **Median Probability Model threshold** ($	ext{PIP} \ge 0.5$).

In [ ]:
# ==============================================================================
# TODO 6: Posterior Inclusion Probabilities
# 1. Calculate pip as the sample mean of z_draws along axis 0
# 2. Extract selected_features as those with pip >= 0.5
# ==============================================================================
pip = None  # <-- YOUR CODE HERE: np.mean(z_draws, axis=0)

# Build PIP summary table
pip_df = pd.DataFrame({
    'Feature': feature_cols,
    'Inclusion_Probability': pip
}).sort_values(by='Inclusion_Probability', ascending=False).reset_index(drop=True)

# Important features exceeding median probability threshold 0.5
selected_features = pip_df[pip_df['Inclusion_Probability'] >= 0.5]['Feature'].tolist()

print(f"Discovered {len(selected_features)} essential diagnostic features with PIP >= 0.5:")
print(selected_features)
pip_df.head(15).round(4)

In [ ]:
# Horizontal Bar Chart of PIPs
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#c0392b' if p >= 0.5 else '#7f8c8d' for p in pip_df['Inclusion_Probability']]

ax.barh(pip_df['Feature'][::-1], pip_df['Inclusion_Probability'][::-1], color=colors[::-1], edgecolor='black')
ax.axvline(0.5, color='#2c3e50', linestyle='--', linewidth=2, label='Median Model Cutoff (0.5)')
ax.set_xlabel('Posterior Inclusion Probability (PIP)', fontsize=12)
ax.set_title('Wisconsin Diagnostic Breast Cancer: SSVS Feature Inclusion Probabilities', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.legend(frameon=True, loc='lower right')
plt.tight_layout()
plt.show()

---
## 7. Clinical Translation: Odds Ratios & Oncology Decision-Making

In medical diagnostics, logistic regression coefficients $\beta_j$ are on the log-odds scale, which can be difficult for clinicians to interpret directly.

To translate coefficients into clinical terms, we compute the **Odds Ratio (OR)**:
$$\text{Odds} = \frac{P(\text{Malignant})}{P(\text{Benign})} = \exp\left(\beta_0 + \sum_{j=1}^{30} \beta_j x_{ij}\right)$$

Because all predictors were standardized to unit variance, a **1-unit increase in $x_{ij}$** represents a **1-Standard Deviation (1-SD) increase** in that nuclear measurement:
$$\text{Odds Ratio for 1-SD increase in feature } j = \exp(\beta_j)$$

* $\text{OR}_j > 1$: Higher values of this trait significantly **increase** the odds of cancer.
* $\text{OR}_j < 1$: Higher values are associated with benign masses.

---

### ✏️ Exercise 7: Posterior Odds Ratios & Oncology Reflection
#### 💡 Your Task:
Compute the posterior mean and $95\%$ equal-tailed credible intervals for the odds ratios of the effective coefficients.

In [ ]:
# ==============================================================================
# TODO 7: Posterior Odds Ratios
# Compute odds ratios for features (beta_eff_draws[:, 1:])
# 1. Calculate post_mean_beta, ci_lower_beta, ci_upper_beta
# 2. Compute or_draws = np.exp(beta_eff_draws[:, 1:])
# 3. Calculate post_mean_or, ci_lower_or, ci_upper_or
# ==============================================================================
post_mean_beta = None  # <-- YOUR CODE HERE
or_draws = None        # <-- YOUR CODE HERE
post_mean_or = None    # <-- YOUR CODE HERE
ci_lower_or = None     # <-- YOUR CODE HERE
ci_upper_or = None     # <-- YOUR CODE HERE

results_summary = pd.DataFrame({
    'Feature': feature_cols,
    'PIP': pip,
    'Post_Mean_Beta': post_mean_beta,
    'Odds_Ratio_Mean': post_mean_or,
    'OR_95%_CI': [f"[{l:.2f}, {u:.2f}]" for l, u in zip(ci_lower_or, ci_upper_or)]
}).sort_values(by='PIP', ascending=False).reset_index(drop=True)

print("=== TOP DIAGNOSTIC BIOMARKERS & CLINICAL ODDS RATIOS ===")
results_summary.head(12).round(3)

---
## 8. Clinical Interpretation & Guided Discussion

Take 5 minutes with your partner to discuss and write down your answers to the following 3 clinical questions:

### 🩺 Question 1: What does an Odds Ratio of ~14.5 for `texture1` mean to an oncologist?
*(Write your thoughts below: How does texture reflect chromatin irregularity in tumor biology?)*

> **Your Answer**: 

### 🩺 Question 2: Why are `radius3` (worst radius) and `concavity3` prioritized over `radius1` (mean radius)?
*(Write your thoughts below: Why do the 3 largest cells matter more than the slide average?)*

> **Your Answer**: 

### 🩺 Question 3: Why is posterior uncertainty quantification critical in medical AI?
*(Write your thoughts below: Why are point predictions dangerous when false negatives are life-threatening?)*

> **Your Answer**: 